### 1. Evaluation for assessing embedding models and semantic search

#### 1.1 Top-k Retrieval Accuracy

Measures how often the correct answer appears in the top-k most similar results for a query. Higher accuracy means better retrieval performance.    

In [3]:
from sentence_transformers import util
import torch

def top_k_accuracy(model, queries, expected_rule_ids, rule_embeddings, dataset, k=5):
    correct = 0
    for query, expected_id in zip(queries, expected_rule_ids):
        query_emb = model.encode(query, normalize_embeddings=True)
        cos_scores = util.cos_sim(query_emb, rule_embeddings)[0]
        top_k_indices = torch.topk(cos_scores, k=min(k, len(dataset))).indices
        top_k_rule_ids = [dataset[idx]["rule_id"] for idx in top_k_indices]
        if expected_id in top_k_rule_ids:
            correct += 1
    accuracy = correct / len(queries)
    print(f"Top-{k} retrieval accuracy: {accuracy:.2f}")

#### 1.2 Mean Reciprocal Rank (MRR)

Calculates the average inverse rank of the first correct answer for each query. Higher MRR means correct answers are ranked closer to the top.

In [4]:
from sentence_transformers import util
import torch

def mean_reciprocal_rank(model, queries, expected_rule_ids, rule_embeddings, dataset):
    ranks = []
    for query, expected_id in zip(queries, expected_rule_ids):
        query_emb = model.encode(query, normalize_embeddings=True)
        cos_scores = util.cos_sim(query_emb, rule_embeddings)[0]
        sorted_indices = torch.argsort(cos_scores, descending=True)
        sorted_rule_ids = [dataset[idx]["rule_id"] for idx in sorted_indices]
        rank = sorted_rule_ids.index(expected_id) + 1 if expected_id in sorted_rule_ids else len(sorted_rule_ids)
        ranks.append(1.0 / rank)
    mrr = sum(ranks) / len(ranks)
    print(f"Mean Reciprocal Rank (MRR): {mrr:.2f}")

#### 1.3 Clustering Quality (Silhouette Score)

Assesses how well embeddings group similar items together using clustering. A higher silhouette score means better-defined and more meaningful clusters.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

def clustering_quality(rule_embeddings, n_clusters=5):
    X = np.vstack(rule_embeddings)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    print(f"Silhouette Score: {score:.2f}")

### 2. Evaluation for summarization

#### 2.1 ROUGE

measures overlap between generated and reference summaries

In [9]:
from rouge_score import rouge_scorer

reference = "Baseline chilled water design supply temperature shall be modeled at 44F."
summary = "Chilled water supply temperature is set to 44F in the baseline model."

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = scorer.score(reference, summary)
print(scores)

{'rouge1': Score(precision=0.5833333333333334, recall=0.6363636363636364, fmeasure=0.6086956521739131), 'rouge2': Score(precision=0.18181818181818182, recall=0.2, fmeasure=0.1904761904761905), 'rougeL': Score(precision=0.4166666666666667, recall=0.45454545454545453, fmeasure=0.43478260869565216)}


### 2.2 BLEU 

measures n-gram overlap, less common for summarization

In [8]:
from nltk.translate.bleu_score import sentence_bleu

reference = ["Baseline chilled water design supply temperature shall be modeled at 44F.".split()]
summary = "Chilled water supply temperature is set to 44F in the baseline model.".split()

bleu_score = sentence_bleu(reference, summary)
print("BLEU score:", bleu_score)

BLEU score: 5.791739854583281e-155


c:\_RCT_fixed\venv\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
c:\_RCT_fixed\venv\lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
